Import gw data analyzed

In [46]:
import os
import pandas as pd

# Load the csv file
league_name = 'ifc' #'rpk', 'ifc', 'rbsc'

In [47]:
def get_dim_manager(league_name):
    # 1. Define the relative directory path
    folder_path = f"../data/2025-2026/{league_name}/"
    dim_manager_path = os.path.join(folder_path, "dim_managers.csv")
    return pd.read_csv(dim_manager_path)

def get_df(league_name):
    # 1. Define the relative directory path
    folder_path = f"../data/2025-2026/{league_name}/"

    # 2. Load all available files (assuming up to gw36 are available right now)
    # We loop from 1 to 36 to build the full historic dataset
    file_names = [
        os.path.join(folder_path, f"gw{str(i).zfill(2)}_analyzed.csv")
        for i in range(1, 38+1)
    ]

    # Filter out files that don't exist yet (safeguard for future weeks like gw37, gw38)
    existing_files = [f for f in file_names if os.path.exists(f)]
    if not existing_files:
        raise FileNotFoundError(
            f"No analyzed CSV files found in directory: {folder_path}"
        )

    # 3. Read and combine all found CSV files into one master DataFrame
    df = pd.concat([pd.read_csv(file) for file in existing_files], ignore_index=True)

    # 4. Join and get manager info
    dim_manager_df = get_dim_manager(league_name)
    df = df.merge(
        dim_manager_df[["id", "player_first_name", "player_last_name", "name"]],
        left_on="manager_id", right_on="id",
        how="left"
    )

    column_order = [
        'manager_id',
        'id',
        'player_first_name',
        'player_last_name',
        'name',
        'gw_no',
        'points',
        'transfers_cost',
        'active_chip',
        'points_on_bench',
        'h2h_points',
        'rank',
        'pnl',
    ]

    return df[column_order]

In [48]:
df = get_df(league_name)

In [49]:
df

,manager_id,id,player_first_name,player_last_name,name,gw_no,points,transfers_cost,active_chip,points_on_bench,h2h_points,rank,pnl
0,8055872,8055872,Sangdaet,Wanichanan,Sang,1,70,0,NaN,12,70,1,300
1,414123,414123,Charlie,Ryan,Atlético Nicotinho,1,67,0,NaN,5,67,2,0
2,5906816,5906816,Thanachai,Kittichokwattana,Birdfever,1,65,0,NaN,8,65,3,0
3,1879361,1879361,Norawit,Hempornwisarn,Norawich City,1,64,0,NaN,6,64,4,0
4,1156773,1156773,Kongkrit,D.,moooooonited,1,60,0,NaN,1,60,5,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
679,5906816,5906816,Thanachai,Kittichokwattana,Birdfever,38,44,0,NaN,0,44,13,0
680,1671397,1671397,Kittikun,Deeya,Tongnajababy,38,43,0,NaN,0,43,14,0
681,2567689,2567689,Yuthakarn,Phithaksung,EnerGY_RoyalFlush.tv,38,41,0,NaN,0,41,15,-100
682,8055872,8055872,Sangdaet,Wanichanan,Sang,38,39,0,NaN,3,39,16,-100


In [50]:
# check 38 entries for every player
for no_of_entries in df.groupby('manager_id')['h2h_points'].count():
    assert no_of_entries == 38

In [51]:
def get_player_history(df, manager_id, gw_start=1, gw_end=38, chronological=True):
    """Extracts and filters the gameweek history for a single manager."""
    # 1. Filter for the specific manager and gameweek range
    player_df = df[
        (df["manager_id"] == manager_id)
        & (df["gw_no"] >= gw_start)
        & (df["gw_no"] <= gw_end)
    ].copy()

    # 2. Sort by gameweek order
    sort_order = True if chronological else False
    player_df = player_df.sort_values(by="gw_no", ascending=sort_order)

    # 3. Reorder columns to make it readable as a personal timeline
    column_order = [
        "gw_no",
        "rank",
        "h2h_points",
        "points",
        "points_on_bench",
        "transfers_cost",
        "active_chip",
        "pnl",
    ]

    # Clean up the index so it looks like a fresh dataframe (0, 1, 2...)
    return player_df[column_order].reset_index(drop=True)

In [52]:
# get_player_history(df, 1357701)

In [53]:
def summarize_league(league_name, gw_start=1, gw_end=38):
    """Loads all available CSV files for a specific league, computes performance

    statistics, and filters the final summary by the requested gameweek range.
    """
    df = get_df(league_name)

    # 4. Calculate Weekly Winners and Losers across the whole dataset
    max_ranks = df.groupby("gw_no")["rank"].transform("max")
    df["is_weekly_winner"] = df["rank"] == 1
    if league_name == "rbsc":
        df["is_weekly_loser"] = df["rank"] == max_ranks
    else:
        df["is_weekly_loser"] = df["rank"] >= max_ranks - 2

    # 5. NOW apply your custom gameweek range filter
    df_filtered = df[(df["gw_no"] >= gw_start) & (df["gw_no"] <= gw_end)].copy()

    # 6. Aggregate metrics per manager for the filtered range
    summary = (
        df_filtered.groupby("manager_id")
        .agg(
            total_h2h_points=("h2h_points", "sum"),
            total_points=("points", "sum"),
            bench_points=("points_on_bench", "sum"),
            total_pnl=("pnl", "sum"),
            weekly_wins=("is_weekly_winner", "sum"),
            weekly_losses=("is_weekly_loser", "sum"),
        )
        .reset_index()
    )

    # 7. Add Dense Rank based on H2H points within this specific window
    summary["overall_rank"] = (
        summary["total_h2h_points"]
        .rank(method="dense", ascending=False)
        .astype(int)
    )

    # 8. Add Meta info to keep track of the slice
    summary["gw_start"] = gw_start
    summary["gw_end"] = gw_end

    # 9. Dynamically load and merge Manager Names from the same folder
    dim_manager_df = get_dim_manager(league_name)
    summary = summary.merge(
        dim_manager_df[["id", "player_first_name", "player_last_name", "name"]],
        left_on="manager_id", right_on="id",
        how="left"
    )

    # 10. Clean layout order
    column_order = [
        "gw_start",
        "gw_end",
        "overall_rank",
        "manager_id",
        "name",
        "player_first_name",
        "player_last_name",
        "total_h2h_points",
        "total_points",
        "bench_points",
        "total_pnl",
        "weekly_wins",
        "weekly_losses",
    ]

    return summary[column_order].sort_values("overall_rank")

In [54]:
summary = summarize_league(league_name)

In [55]:
summary

,gw_start,gw_end,overall_rank,manager_id,name,player_first_name,player_last_name,total_h2h_points,total_points,bench_points,total_pnl,weekly_wins,weekly_losses
0,1,38,1,160759,Bristol Boy FC,Pattanun,Yongvibulsiri,2366,2382,340,-300,0,3
10,1,38,2,3966243,Ethan hunt,nattanan,lertpanyarote,2328,2328,361,900,3,1
2,1,38,3,1156773,moooooonited,Kongkrit,D.,2324,2324,321,100,1,2
4,1,38,4,1879361,Norawich City,Norawit,Hempornwisarn,2320,2320,292,800,3,2
7,1,38,5,2558211,Phuket,phuket,kumhangphon,2288,2312,322,1100,5,6
12,1,38,5,5265858,peerarat,Peerarat,Laolugsanalerd,2288,2292,328,200,2,3
6,1,38,6,2510442,it's cole innit,Pitcha,Lertvinyu,2244,2256,355,300,3,6
5,1,38,7,2412148,ngnteam,Tawiwut,Charuwat,2214,2214,395,800,3,5
9,1,38,8,3723249,MeatballMarinara,Chotiwit,Jiratananuwong,2182,2206,357,0,2,7
13,1,38,9,5385738,kentwins,Thanpisit,Manomaiphibul,2171,2187,373,600,3,7


In [56]:
df.to_csv(f"fact_gw_{league_name}.csv", index=False)

In [57]:
summary.to_csv(f"summary_{league_name}.csv", index=False)

Add end of season prize to summary (RPK)

In [19]:
# add end of season prize 
def add_eos_prize(summary, prize: dict):
    summary_copy = summary.copy()
    # assign prize to rank
    summary_copy['end_of_season_prize'] = summary['overall_rank'].map(prize).fillna(0).astype(int)
    #    assert summary_copy['end_of_season_prize'] == 0 # not applicable to ifc league because everybody paid upfront
    summary_copy['pnl'] = summary_copy['total_pnl'] + summary_copy['end_of_season_prize']
    return summary_copy

In [20]:
# add_eos_prize(summary, {1:int(0.5*19000), 2:int(0.25*19000), 3:int(0.15*19000)}) # ifc
add_eos_prize(summary, {1:3000, 2:2000, 3:1000, 6:-1000, 7:-2000, 8:-3000}) # rpk
# add_eos_prize(summary, {1:int(0.5*19000), 2:int(0.25*19000), 3:int(0.15*19000)}) # rbsc

,gw_start,gw_end,overall_rank,manager_id,name,player_first_name,player_last_name,total_h2h_points,total_points,bench_points,total_pnl,weekly_wins,weekly_losses,end_of_season_prize,pnl
0,1,38,1,294329,Aekk72,Atthapon,Parkart,2301,2305,323,0,6,15,3000,3000
1,1,38,2,967075,Victory Goalkeres,sirawat,dulyavit,2251,2259,250,-200,5,17,2000,1800
4,1,38,3,2412148,ngnteam,Tawiwut,Charuwat,2214,2214,395,2650,13,15,1000,3650
3,1,38,4,1369948,Morty FC,Pattapong,Charoenchaipong,2065,2073,344,-1300,4,26,0,-1300
2,1,38,5,1357701,OnkaewmaneeN,Nithiz,Onkaewmanee,1875,1999,463,-50,7,24,0,-50
5,1,38,6,6149266,pairyn FC,Natthawat,Charoenkitmongkol,1870,1870,74,-1100,5,29,-1000,-2100
